# Suffix SMC: direct validation

This notebook answers one question: **does the SMC-mixture estimate agree with direct sampling when both use the same generation and ARI implementation?**

Run the cells from top to bottom. Only component archives and the direct ARI reference are persisted on Drive so interrupted simulations can resume. Tables, diagnostics, and figures are created directly below the analysis cells and are not written as extra CSV, PNG, or report files.

In [ ]:
# 1. Connect Drive, clone/update the repository, and install dependencies.
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main.git'
REPO_ROOT = Path('/content/rea_of_llms_supplementary_code-main')

if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)
elif REPO_ROOT.exists():
    raise RuntimeError(f'{REPO_ROOT} exists but is not a Git repository.')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Ready:', REPO_ROOT)

In [ ]:
# 2. EDIT PARAMETERS HERE.
# Use True only for stages that should generate new model samples.
RUN_DRIVE_AUDIT = False
RUN_DIRECT_REFERENCE = False
RUN_DIRECT_CALIBRATION = False
RUN_SMC = False
ARI_CONVENTION = 'current'  # 'current' or 'official_public'

# Change this one path if your data lives in a different Drive folder.
OUTPUT_ROOT = Path('/content/drive/MyDrive/rea_tail_probability_validation_shared_A')

COMPONENTS = [
    {'cap': 6,  'lambda': -0.5},
    {'cap': 6,  'lambda': -1.0},
    {'cap': 6,  'lambda': -1.5},
    {'cap': 7,  'lambda': -1.0},
    {'cap': 7,  'lambda': -1.5},
    {'cap': 7,  'lambda': -2.0},
    {'cap': 8,  'lambda': -1.5},
    {'cap': 8,  'lambda': -2.0},
    {'cap': 8,  'lambda': -2.5},
    {'cap': 10, 'lambda': -2.0},
    {'cap': 10, 'lambda': -2.5},
    {'cap': 10, 'lambda': -3.0},
    {'cap': 10, 'lambda': -3.5},
]
SEED_GROUPS = [200, 300]
TAIL_THRESHOLDS = '6,7,8,10'
NUM_PARTICLES = 256
MCMC_STEPS_PER_LEVEL = 10
LAMBDA_STEP = 0.25
ESS_THRESHOLD = 0.5
MAX_NEW_TOKENS = 100
SUFFIX_SIZES = '1,2,4,8,16,32,64,100'
SUFFIX_RHOS = '0.20,0.18,0.16,0.14,0.12,0.10,0.06,0.04'
BATCH_SIZE = 32
DEVICE = 'cuda'

DIRECT_NUM_SAMPLES = 200_000
DIRECT_CALIBRATION_SAMPLES = 200_000
DIRECT_BATCH_SIZE = 256
DIRECT_SEED = 91_000

MODEL_NAME = 'roneneldan/TinyStories-8M'
TOKENIZER_NAME = 'EleutherAI/gpt-neo-125M'
PROMPT = 'Once upon a time, in a big forest, there lived a rhinoc'

print(f'ARI convention:      {ARI_CONVENTION}')
print(f'Read-only audit:     {RUN_DRIVE_AUDIT}')
print(f'New direct sampling: {RUN_DIRECT_REFERENCE}')
print(f'Direct calibration:  {RUN_DIRECT_CALIBRATION}')
print(f'New SMC sampling:    {RUN_SMC}')
print(f'Planned components:  {len(COMPONENTS) * len(SEED_GROUPS)}')
print(f'Data folder:         {OUTPUT_ROOT}')

In [ ]:
# 3. Optional read-only audit of existing Drive artifacts. This cell never writes files.
import pandas as pd
from IPython.display import display
from rea_llms.data_reuse_audit import REUSE_DIMENSIONS, audit_run_directory

if RUN_DRIVE_AUDIT:
    audit_rows = audit_run_directory(
        OUTPUT_ROOT, target_ari_convention=ARI_CONVENTION,
    )
    if not audit_rows:
        print(f'No recognized run artifacts found under {OUTPUT_ROOT}')
    else:
        audit_df = pd.DataFrame(audit_rows)
        inventory_columns = [
            'artifact_type', 'algorithm', 'ari_convention', 'prompt_sha256',
            'model_name', 'tokenizer_name', 'max_new_tokens', 'ari_cap',
            'lambda_final', 'num_particles', 'mcmc_steps_per_level', 'seed',
            'has_raw_ari', 'has_final_smc_weights', 'has_log_z',
            'has_token_ids', 'runtime_seconds', 'generated_token_count',
            'proposal_count', *REUSE_DIMENSIONS, 'path',
        ]
        display(audit_df.reindex(columns=inventory_columns))
        failed = []
        for _, row in audit_df.iterrows():
            reasons = [row.get(f'reason_{key}') for key in REUSE_DIMENSIONS if not row.get(key, False)]
            failed.append('; '.join(reason for reason in reasons if isinstance(reason, str)))
        limitations = audit_df[['path']].copy()
        limitations['limitations'] = failed
        display(limitations)
        print(f'Read-only audit complete: {len(audit_df)} artifacts; no samples generated.')
else:
    print('Drive audit disabled. Set RUN_DRIVE_AUDIT=True to inspect existing artifacts.')


In [ ]:
# 4. Check the runtime and create/load the matching direct reference.
import json
import torch

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('DEVICE=cuda but this Colab runtime has no GPU.')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DIRECT_ROOT = OUTPUT_ROOT / (
    'current_direct_reference' if ARI_CONVENTION == 'current'
    else 'official_public_direct_reference'
)
if RUN_DIRECT_REFERENCE:
    command = [
        sys.executable, '-m', 'rea_llms.current_direct_reference',
        '--model-name', MODEL_NAME, '--tokenizer-name', TOKENIZER_NAME,
        '--prompt', PROMPT, '--device', DEVICE,
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--num-samples', str(DIRECT_NUM_SAMPLES),
        '--batch-size', str(DIRECT_BATCH_SIZE), '--seed', str(DIRECT_SEED),
        '--thresholds', TAIL_THRESHOLDS, '--ari-convention', ARI_CONVENTION,
        '--output-dir', str(DIRECT_ROOT),
        '--skip-completed',
    ]
    subprocess.run(command, check=True)

if RUN_DIRECT_CALIBRATION:
    if ARI_CONVENTION != 'official_public':
        raise RuntimeError("Set ARI_CONVENTION='official_public' for calibration.")
    calibration_root = OUTPUT_ROOT / 'official_public_direct_reference'
    calibration_command = [
        sys.executable, '-m', 'rea_llms.current_direct_reference',
        '--model-name', MODEL_NAME, '--tokenizer-name', TOKENIZER_NAME,
        '--prompt', PROMPT, '--device', DEVICE,
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--num-samples', str(DIRECT_CALIBRATION_SAMPLES),
        '--batch-size', str(DIRECT_BATCH_SIZE), '--seed', str(DIRECT_SEED),
        '--thresholds', TAIL_THRESHOLDS, '--ari-convention', 'official_public',
        '--output-dir', str(calibration_root), '--skip-completed',
    ]
    subprocess.run(calibration_command, check=True)

direct_summary_path = DIRECT_ROOT / 'summary.json'
if not direct_summary_path.is_file():
    raise FileNotFoundError('Direct reference missing. Set RUN_DIRECT_REFERENCE=True and rerun this cell.')
direct_summary = json.loads(direct_summary_path.read_text())
print(f"Direct reference ready: N={direct_summary['statistics']['count']:,}")
for row in direct_summary['statistics']['tails']:
    print(f"  ARI >= {row['threshold']:g}: {row['count']:,} events, p={row['empirical_probability']:.7g}")

In [ ]:
# 5. Generate missing SMC components, or load the exact requested cache.
# This cell does not write reports, tables, or plots to Drive.
from suffix_direct_mh_smc_experiment.run_tail_validation import parse_args, run_validation

arguments = [
    '--model-name', MODEL_NAME, '--tokenizer-name', TOKENIZER_NAME,
    '--prompt', PROMPT, '--device', DEVICE,
    '--components-json', json.dumps(COMPONENTS),
    '--seed-groups', ','.join(map(str, SEED_GROUPS)),
    '--thresholds', TAIL_THRESHOLDS, '--ari-convention', ARI_CONVENTION,
    '--lambda-step', str(LAMBDA_STEP),
    '--num-particles', str(NUM_PARTICLES), '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--mcmc-steps-per-level', str(MCMC_STEPS_PER_LEVEL),
    '--ess-threshold', str(ESS_THRESHOLD), '--suffix-sizes', SUFFIX_SIZES,
    '--suffix-rhos', SUFFIX_RHOS, '--batch-size', str(BATCH_SIZE),
    '--output-root', str(OUTPUT_ROOT),
    '--current-direct-summary', str(direct_summary_path), '--skip-completed-runs',
]
if not RUN_SMC:
    arguments.append('--audit-only')

validation = run_validation(
    parse_args(arguments),
    persist_analysis=False,
    fast_cache=True,
)

status = validation.component_status.groupby(['seed_group', 'status']).size().unstack(fill_value=0)
print('Component cache:')
print(status.to_string())
print('Complete seed groups used:', sorted(validation.complete_grouped))
if not validation.complete_grouped:
    raise RuntimeError('No complete seed group is available for mixture estimation.')

In [ ]:
# 6. Simple numerical results.
import numpy as np
import pandas as pd
from IPython.display import display

aggregate = validation.comparison.query("row_type == 'aggregate'").copy()
results = aggregate[[
    'threshold', 'current_direct_event_count', 'current_direct_probability',
    'mixture_probability', 'current_direct_wilson_95_lower',
    'current_direct_wilson_95_upper', 'weight_ess_over_n',
]].rename(columns={
    'threshold': 'ARI threshold',
    'current_direct_event_count': 'direct events',
    'current_direct_probability': 'direct probability',
    'mixture_probability': 'SMC-mixture probability',
    'weight_ess_over_n': 'correction ESS/N',
})
results['SMC / direct'] = results['SMC-mixture probability'] / results['direct probability']
results['relative difference'] = results['SMC / direct'] - 1.0
results = results[[
    'ARI threshold', 'direct events', 'direct probability',
    'SMC-mixture probability', 'SMC / direct', 'relative difference',
    'correction ESS/N', 'current_direct_wilson_95_lower',
    'current_direct_wilson_95_upper',
]]
display(results.style.format({
    'ARI threshold': '{:.0f}', 'direct events': '{:,.0f}',
    'direct probability': '{:.7g}', 'SMC-mixture probability': '{:.7g}',
    'SMC / direct': '{:.2f}', 'relative difference': '{:+.1%}',
    'correction ESS/N': '{:.3f}',
    'current_direct_wilson_95_lower': '{:.7g}',
    'current_direct_wilson_95_upper': '{:.7g}',
}))

print(f"Direct completions: {int(aggregate.iloc[0]['current_direct_sample_count']):,}")
print(f"Complete independent seed groups: {len(validation.complete_grouped)}")
print(f"SMC components in each complete group: {len(validation.specs)}")
print(f"Mean correction ESS/N: {aggregate['weight_ess_over_n'].mean():.3f}")
print(f"Mean SMC runtime per complete group: {aggregate['runtime_seconds'].mean() / 60:.1f} minutes")

In [ ]:
# 7. Inline visual summary. No image files are created.
import matplotlib.pyplot as plt
from rea_llms.official_direct_data import boxplot_statistics

plot_rows = aggregate.sort_values('threshold')
x = plot_rows['threshold'].to_numpy(float)
direct_p = plot_rows['current_direct_probability'].to_numpy(float)
smc_p = plot_rows['mixture_probability'].to_numpy(float)
direct_low = plot_rows['current_direct_wilson_95_lower'].to_numpy(float)
direct_high = plot_rows['current_direct_wilson_95_upper'].to_numpy(float)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

ax = axes[0, 0]
ax.errorbar(x, direct_p, yerr=[direct_p-direct_low, direct_high-direct_p],
            marker='o', capsize=4, label='direct', color='#2C7BB6')
ax.plot(x, smc_p, marker='s', label='SMC mixture', color='#D95F02')
ax.set_yscale('log'); ax.set_xticks(x); ax.set_xlabel('raw ARI threshold')
ax.set_ylabel('tail probability'); ax.set_title('Base-model tail probability')
ax.grid(alpha=.25, which='both'); ax.legend()

ax = axes[0, 1]
ratios = smc_p / direct_p
colors = ['#3A923A' if 0.8 <= value <= 1.2 else '#D95F02' for value in ratios]
ax.bar(x.astype(str), ratios, color=colors); ax.axhline(1, color='black', linewidth=1)
ax.set_xlabel('raw ARI threshold'); ax.set_ylabel('SMC / direct')
ax.set_title('Agreement with direct sampling'); ax.grid(axis='y', alpha=.25)

ax = axes[1, 0]
boxes = []
direct_ari_path = DIRECT_ROOT / 'direct_ari.npz'
with np.load(direct_ari_path, allow_pickle=False) as archive:
    direct_ari = archive['raw_ari']
stats = boxplot_statistics(direct_ari)
boxes.append({'label': 'direct', 'q1': stats['q1'], 'med': stats['median'],
              'q3': stats['q3'], 'whislo': stats['whisker_low'],
              'whishi': stats['whisker_high'], 'fliers': []})
for index, spec in enumerate(validation.specs):
    rows = [components[index] for components in validation.complete_grouped.values()]
    values = np.concatenate([row.raw_ari for row in rows])
    log_weights = np.concatenate([row.final_log_weights - np.log(len(rows)) for row in rows])
    weights = np.exp(log_weights - np.logaddexp.reduce(log_weights))
    stats = boxplot_statistics(values, weights=weights)
    boxes.append({'label': f"c={spec.cap:g}\nlam={spec.lambda_final:g}",
                  'q1': stats['q1'], 'med': stats['median'], 'q3': stats['q3'],
                  'whislo': stats['whisker_low'], 'whishi': stats['whisker_high'],
                  'fliers': []})
artists = ax.bxp(boxes, showfliers=False, patch_artist=True)
artists['boxes'][0].set_facecolor('#2C7BB6')
for box in artists['boxes'][1:]: box.set_facecolor('#FDAE61')
ax.set_ylabel('raw ARI'); ax.set_title('Direct and tilted terminal populations')
ax.tick_params(axis='x', labelsize=8); ax.grid(axis='y', alpha=.25)

ax = axes[1, 1]
diagnostics = validation.component_diagnostics.drop_duplicates('run_id')
for cap, rows in diagnostics.groupby('cap'):
    summary = rows.groupby('lambda_final', as_index=False)['corrected_ess_over_n'].mean()
    ax.plot(summary['lambda_final'], summary['corrected_ess_over_n'], marker='o', label=f'cap {cap:g}')
ax.axhline(.1, color='#D73027', linestyle='--', linewidth=1, label='ESS/N = 0.1')
ax.set_xlabel('terminal lambda'); ax.set_ylabel('correction ESS/N')
ax.set_title('Importance-correction stability'); ax.grid(alpha=.25); ax.legend()

fig.suptitle('Suffix SMC validation summary', fontsize=16)
fig.tight_layout()
plt.show()

print('Interpretation:')
print('- The probability and ratio panels compare SMC with the matching direct implementation.')
print('- The boxplots show tilted SMC targets; they are not reconstructed base-model distributions.')
print('- Low correction ESS/N means that a tail estimate may be unstable even if rare ARI values were discovered.')
if len(validation.complete_grouped) == 1:
    print('- Only one complete seed group is available, so between-seed stability is not yet measured.')
if int(results.loc[results['ARI threshold'].idxmax(), 'direct events']) < 20:
    print('- The highest threshold has fewer than 20 direct events and remains noisy.')